# Master Benchmark Engine v2 (LungRay Production CADx Platform)
## Khảo Sát Đối Chiếu Hiệu Năng Đa Kiến Trúc (Multi-Architecture Comparative Analysis)
### 4 Checkpoints + 2 Biến Thể Ensemble trên Tập Kiểm Định Y Tế Độc Lập

---

### Mục tiêu khảo nghiệm khoa học:
1. **Head A (14 bệnh lồng ngực NIH ChestX-ray14)**: Đánh giá trên toàn bộ 25,596 ảnh test chính thức chia theo `Patient_ID` (Zero Patient Leakage).
2. **Head B (Sàng lọc Lao phổi TB Triage)**: Đánh giá nhị phân trên tập test độc lập, khử trùng lặp đường dẫn tuyệt đối.
3. **Head C (Sàng lọc COVID-19 Nhị phân)**: Đánh giá trên tập test chuẩn y tế, phân loại nhị phân (Âm tính vs COVID-19).
4. **WSOL Grounding (Grad-CAM Localization)**: Đo lường Mean Box-IoU và Success Rate@0.30, @0.50 so với Bounding Box chuyên gia RSNA Pneumonia.
5. **Chuỗi Nguồn Gốc (Provenance)**: Tự động tính mã băm SHA-256 cho toàn bộ weights, xuất `test_predictions_raw.csv.gz` và khoảng tin cậy 95% Bootstrap CI.
6. **Ma trận Quyết định Tuyển dụng**: So sánh đối chiếu để đưa ra khuyến nghị chiến lược (Dừng lại ghi CV vs Tiếp tục Fine-tune).


In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Environment Setup, Dependencies & Hardware Discovery
# ---------------------------------------------------------------------------
import os
import sys
import glob
import time
import json
import hashlib
import gzip
import random
import re
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import timm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Execution Hardware: {device}")
if torch.cuda.is_available():
    print(f" -> Device Name: {torch.cuda.get_device_name(0)}")
    print(f" -> Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print(" -> CUDA unavailable; this benchmark is intended for a Kaggle GPU session.")

OUTPUT_DIR = Path("/kaggle/working/benchmark_v2_results")
FIGURES_DIR = Path("/kaggle/working/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = 256
BATCH_SIZE = 64
NUM_WORKERS = min(4, os.cpu_count() or 1)

NIH_14_DISEASES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule",
    "Pneumonia", "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia",
]
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
print(f"[+] Output Directory: {OUTPUT_DIR}")
print(f"[+] Figure Directory: {FIGURES_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: Dataset Discovery, Zero-Leakage Test Splits & DataLoaders
# ---------------------------------------------------------------------------
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".dcm")
k_input = "/kaggle/input" if Path("/kaggle/input").is_dir() else str(Path("dataset").resolve())
print(f"[*] Scanning datasets under {k_input}...")


def find_subdirs_by_keywords(base_path: str, keywords: List[str]) -> List[str]:
    matches, seen = [], set()
    if not os.path.isdir(base_path):
        return matches
    for root, dirs, _ in os.walk(base_path):
        canonical = os.path.normcase(os.path.abspath(root))
        if canonical in seen:
            dirs[:] = []
            continue
        seen.add(canonical)
        dirs.sort()
        if any(key.casefold() in os.path.basename(root).casefold() for key in keywords):
            matches.append(root)
    return sorted(set(matches))


def build_filename_index(search_dirs: List[str], extensions=IMAGE_EXTENSIONS) -> Dict[str, str]:
    """Index supported image files once, with deterministic preference for test paths."""
    index, rank_by_name, seen_dirs = {}, {}, set()
    allowed = {ext.casefold() for ext in extensions}
    for search_dir in search_dirs:
        if not search_dir or not os.path.isdir(search_dir):
            continue
        for root, dirs, files in os.walk(search_dir):
            canonical = os.path.normcase(os.path.abspath(root))
            if canonical in seen_dirs:
                dirs[:] = []
                continue
            seen_dirs.add(canonical)
            dirs.sort()
            for filename in sorted(files):
                if Path(filename).suffix.casefold() not in allowed:
                    continue
                key = filename.casefold()
                priority = 0 if "test" in Path(root).parts[-1].casefold() else 1
                if key not in index or priority < rank_by_name[key]:
                    index[key] = os.path.join(root, filename)
                    rank_by_name[key] = priority
    return index


def read_cxr_rgb(path: str, img_size: int = IMG_SIZE) -> np.ndarray:
    """Read PNG/JPEG or DICOM, returning an RGB uint8 image at the requested size."""
    if Path(path).suffix.casefold() == ".dcm":
        import pydicom
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            raise ValueError(f"DICOM image has no finite pixels: {path}")
        lo, hi = np.percentile(finite, [0.5, 99.5])
        if hi <= lo:
            hi = lo + 1.0
        arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
        if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
            arr = 1.0 - arr
        gray = np.round(arr * 255.0).astype(np.uint8)
        gray = cv2.resize(gray, (img_size, img_size), interpolation=cv2.INTER_AREA)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    with Image.open(path) as source:
        image = source.convert("RGB").resize((img_size, img_size), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.uint8)


class CXRDataset(Dataset):
    def __init__(self, samples, img_size=IMG_SIZE, multi_label=False):
        self.samples = samples
        self.img_size = img_size
        self.multi_label = multi_label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label, filename = self.samples[index]
        rgb = read_cxr_rgb(path, self.img_size).astype(np.float32) / 255.0
        normalized = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(normalized.transpose(2, 0, 1).copy()).float()
        dtype = torch.float32
        target = torch.as_tensor(label, dtype=dtype)
        return tensor, target, filename


def covid_binary_label(raw):
    token = " ".join(str(raw).strip().casefold().replace("_", " ").replace("-", " ").split())
    compact = token.replace(" ", "")
    if token in {"0", "0.0", "false", "no", "normal", "negative", "neg", "non covid", "noncovid", "pneumonia", "viral pneumonia", "lung opacity"}:
        return 0
    if any(term in compact for term in ("noncovid", "notcovid", "covidnegative")):
        return 0
    if token in {"1", "1.0", "true", "yes", "positive", "pos", "covid", "covid 19", "covid19", "sars cov 2"}:
        return 1
    if compact in {"covidpositive", "positivecovid", "covid19positive", "positivecovid19", "sarscov2positive"}:
        return 1
    if any(term in compact for term in ("normal", "pneumonia", "opacity", "negative")):
        return 0
    return None


nih_candidates = find_subdirs_by_keywords(k_input, ["nih", "chestxray", "chest-x-ray"])
tb_candidates = find_subdirs_by_keywords(k_input, ["tuberculosis", "tb"])
covid_candidates = find_subdirs_by_keywords(k_input, ["covid", "covidx"])
rsna_candidates = find_subdirs_by_keywords(k_input, ["rsna", "pneumonia"])
print("[*] Candidate roots:", {"NIH": nih_candidates, "TB": tb_candidates, "COVID": covid_candidates, "RSNA": rsna_candidates})

# NIH official held-out split: test_list.txt takes precedence; otherwise require split=test.
nih_index = build_filename_index(nih_candidates)
nih_meta_candidates = sorted(set(
    glob.glob(f"{k_input}/**/Data_Entry_2017*.csv", recursive=True)
    + glob.glob(f"{k_input}/**/patient_splits.csv", recursive=True)
    + ["dataset/nih-chestXray-14/Data_Entry_2017.csv", "dataset/splits/patient_splits.csv"]
))
nih_meta_path = next((candidate for candidate in nih_meta_candidates if os.path.isfile(candidate)), None)
nih_df = pd.read_csv(nih_meta_path) if nih_meta_path else None
nih_test_list_candidates = sorted(set(glob.glob(f"{k_input}/**/test_list.txt", recursive=True) + ["dataset/nih-chestXray-14/test_list.txt"]))
nih_test_filenames = set()
nih_test_list_path = next((candidate for candidate in nih_test_list_candidates if os.path.isfile(candidate) and "covid" not in candidate.casefold() and "rsna" not in candidate.casefold()), None)
if nih_test_list_path:
    with open(nih_test_list_path, encoding="utf-8", errors="replace") as handle:
        nih_test_filenames = {line.strip() for line in handle if line.strip()}

nih_samples = []
if nih_df is not None and nih_index:
    filename_col = next((col for col in ("Image Index", "filename", "image_id") if col in nih_df.columns), None)
    label_col = next((col for col in ("Finding Labels", "labels", "label") if col in nih_df.columns), None)
    if filename_col and label_col:
        if nih_test_filenames:
            nih_test_df = nih_df[nih_df[filename_col].astype(str).isin(nih_test_filenames)]
        elif "split" in nih_df.columns:
            nih_test_df = nih_df[nih_df["split"].astype(str).str.casefold().eq("test")]
        else:
            raise RuntimeError("NIH metadata found without test_list.txt or patient-level split=test; refusing to evaluate a leaky split.")
        for _, row in nih_test_df.iterrows():
            filename = os.path.basename(str(row[filename_col]))
            path = nih_index.get(filename.casefold())
            if not path:
                continue
            raw_labels = set(str(row[label_col]).split("|"))
            label = np.array([float(name in raw_labels) for name in NIH_14_DISEASES], dtype=np.float32)
            nih_samples.append((path, label, filename))

nih_loader = None
ds_nih = CXRDataset(nih_samples, multi_label=True)
if nih_samples:
    if nih_test_filenames and len(nih_test_filenames) != 25596:
        print(f"[!] NIH test_list.txt contains {len(nih_test_filenames):,} entries (expected 25,596 for the full release).")
    nih_loader = DataLoader(ds_nih, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
print(f"[+] NIH official test samples resolved: {len(ds_nih):,}")

# TB folders: deduplicate by filename, then form a deterministic balanced test sample.
tb_by_filename = {}
seen_tb_dirs = set()
for tb_root in tb_candidates:
    for root, dirs, files in os.walk(tb_root):
        canonical = os.path.normcase(os.path.abspath(root))
        if canonical in seen_tb_dirs:
            dirs[:] = []
            continue
        seen_tb_dirs.add(canonical)
        dirs.sort()
        class_name = os.path.basename(root).casefold().replace("_", " ").replace("-", " ").strip()
        label = 1.0 if class_name in {"tuberculosis", "tb"} else 0.0 if class_name == "normal" else None
        if label is None:
            continue
        for filename in sorted(files):
            if Path(filename).suffix.casefold() not in (".png", ".jpg", ".jpeg"):
                continue
            tb_by_filename.setdefault(filename.casefold(), (os.path.join(root, filename), label, filename))
tb_positive = [sample for sample in tb_by_filename.values() if sample[1] == 1.0]
tb_negative = [sample for sample in tb_by_filename.values() if sample[1] == 0.0]
rng = np.random.default_rng(42)
rng.shuffle(tb_positive)
rng.shuffle(tb_negative)
tb_test_samples = tb_positive[:400] + tb_negative[:400]
rng.shuffle(tb_test_samples)
tb_loader = DataLoader(CXRDataset(tb_test_samples), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()) if tb_test_samples else None
print(f"[+] TB test sample counts: total={len(tb_test_samples)}, positive={sum(s[1] for s in tb_test_samples):.0f}, negative={sum(1-s[1] for s in tb_test_samples):.0f}")

# COVIDx CXR-4 test manifest is authoritative; folders are a fallback only.
covid_search_roots = covid_candidates if covid_candidates else [k_input]
covid_file_index = build_filename_index(covid_search_roots)
manifest_candidates = sorted(set(
    glob.glob(f"{k_input}/**/test.txt", recursive=True)
    + glob.glob(f"{k_input}/**/*test*.txt", recursive=True)
    + glob.glob(f"{k_input}/**/covid_splits.csv", recursive=True)
    + ["dataset/splits/covid_splits.csv"]
), key=lambda path: (0 if Path(path).name.casefold() == "test.txt" else 1, path.casefold()))
manifest_candidates = [
    path for path in manifest_candidates
    if os.path.isfile(path) and "test_list.txt" not in path.casefold()
    and "nih" not in path.casefold()
    and ("covid" in path.casefold() or "covidx" in path.casefold() or path.casefold().endswith("covid_splits.csv"))
]


def resolve_covid_file(value):
    basename = os.path.basename(str(value).strip().strip("\"'"))
    return covid_file_index.get(basename.casefold())


def parse_covid_manifest(path):
    samples = []
    if Path(path).suffix.casefold() == ".csv":
        frame = pd.read_csv(path)
        if "split" in frame.columns:
            test_mask = frame["split"].astype(str).str.casefold().eq("test")
            if test_mask.any():
                frame = frame[test_mask]
            elif "test" in Path(path).name.casefold():
                pass
            else:
                return samples
        filename_col = next((col for col in ("filename", "Image Index", "image_id", "file", "path") if col in frame.columns), frame.columns[0] if len(frame.columns) else None)
        label_col = next((col for col in ("label", "Finding Labels", "target", "class", "class_name") if col in frame.columns), None)
        if not filename_col or not label_col:
            return samples
        for _, row in frame.iterrows():
            resolved = resolve_covid_file(row[filename_col])
            label = covid_binary_label(row[label_col])
            if resolved and label is not None:
                samples.append((resolved, float(label), os.path.basename(resolved)))
        return samples

    with open(path, encoding="utf-8", errors="replace") as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts or parts[0].startswith("#"):
                continue
            image_path = next((resolve_covid_file(part) for part in parts if resolve_covid_file(part)), None)
            if image_path is None:
                continue
            label = next((value for part in parts if (value := covid_binary_label(part)) is not None and os.path.basename(part).casefold() != os.path.basename(image_path).casefold()), None)
            if label is not None:
                samples.append((image_path, float(label), os.path.basename(image_path)))
    return samples


covid_manifest_used = None
covid_samples = []
for manifest_path in manifest_candidates:
    parsed = parse_covid_manifest(manifest_path)
    if parsed:
        covid_samples = parsed
        covid_manifest_used = manifest_path
        print(f"[+] COVID test labels loaded from manifest: {manifest_path} ({len(parsed):,} resolved rows)")
        break

if not covid_samples:
    print("[!] No usable COVID test manifest found; falling back to explicit class folders/filenames.")
    folder_labels = {
        "covid": 1, "covid 19": 1, "positive": 1,
        "normal": 0, "negative": 0, "non covid": 0,
        "pneumonia": 0, "viral pneumonia": 0, "lung opacity": 0,
    }
    seen_covid_dirs = set()
    for root in covid_search_roots:
        for parent, dirs, files in os.walk(root):
            canonical = os.path.normcase(os.path.abspath(parent))
            if canonical in seen_covid_dirs:
                dirs[:] = []
                continue
            seen_covid_dirs.add(canonical)
            dirs.sort()
            folder = os.path.basename(parent).casefold().replace("_", " ").replace("-", " ").strip()
            label = folder_labels.get(folder)
            for filename in sorted(files):
                if Path(filename).suffix.casefold() not in (".png", ".jpg", ".jpeg"):
                    continue
                item_label = label
                if item_label is None:
                    item_label = covid_binary_label(Path(filename).stem)
                if item_label is not None:
                    full_path = os.path.join(parent, filename)
                    covid_samples.append((full_path, float(item_label), filename))

# Negative-control fallback is explicit and recorded; it does not silently relabel COVIDx images.
control_source = None
unique_covid = {}
for sample in covid_samples:
    unique_covid.setdefault(os.path.normcase(os.path.abspath(sample[0])), sample)
covid_samples = list(unique_covid.values())
if any(sample[1] == 1.0 for sample in covid_samples) and not any(sample[1] == 0.0 for sample in covid_samples):
    normal_controls = [sample for sample in tb_by_filename.values() if sample[1] == 0.0]
    if normal_controls:
        covid_samples.extend((path, 0.0, f"TB-Normal::{filename}") for path, _, filename in normal_controls)
        control_source = "TB Normal folders"
    elif nih_samples:
        nih_controls = [sample for sample in nih_samples if not np.any(sample[1])]
        covid_samples.extend((path, 0.0, f"NIH-No-Finding::{filename}") for path, _, filename in nih_controls)
        control_source = "NIH No Finding"
    if control_source:
        print(f"[!] COVID manifest had no negative class; using explicit external negative controls from {control_source}.")

positive_covid = [sample for sample in covid_samples if sample[1] == 1.0]
negative_covid = [sample for sample in covid_samples if sample[1] == 0.0]
rng = np.random.default_rng(42)
rng.shuffle(positive_covid)
rng.shuffle(negative_covid)
positive_covid = positive_covid[:400]
negative_covid = negative_covid[:400]
assert positive_covid and negative_covid, (
    "COVID test data must resolve at least one positive and one negative sample. "
    f"manifest={covid_manifest_used!r}, positive={len(positive_covid)}, negative={len(negative_covid)}"
)
covid_test_samples = positive_covid + negative_covid
rng.shuffle(covid_test_samples)
covid_loader = DataLoader(CXRDataset(covid_test_samples), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
print(f"[+] COVID test sample counts: total={len(covid_test_samples)}, positive={len(positive_covid)}, negative={len(negative_covid)}; control_source={control_source or 'manifest/folder labels'}")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Unified Multi-Task Architecture & Checkpoint Discovery
# ---------------------------------------------------------------------------
class MultiTaskChestXrayNet(nn.Module):
    """Checkpoint-compatible LungRay 3-head model used by the Kaggle reference engine."""
    def __init__(self, backbone_name="densenet121", pretrained=False,
                 num_classes_a=14, num_classes_b=1, num_classes_c=1):
        super().__init__()
        self.backbone_name = backbone_name.casefold()
        self.backbone = timm.create_model(self.backbone_name, pretrained=pretrained, num_classes=0)
        self.feature_dim = self.backbone.num_features
        self.feature_norm = nn.LayerNorm(self.feature_dim, elementwise_affine=False)
        self.dropout = nn.Dropout(p=0.2)
        self.head_a = nn.Linear(self.feature_dim, num_classes_a)
        self.head_b = nn.Linear(self.feature_dim, num_classes_b)
        self.head_c = nn.Linear(self.feature_dim, num_classes_c)

    def forward(self, x):
        features = self.backbone(x)
        if features.ndim > 2:
            features = torch.flatten(features, 1)
        features = self.dropout(self.feature_norm(features))
        return {
            "head_a": self.head_a(features),
            "head_b": self.head_b(features),
            "head_c": self.head_c(features),
        }


def checkpoint_state(checkpoint_path):
    try:
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:  # PyTorch releases before weights_only was added.
        payload = torch.load(checkpoint_path, map_location="cpu")
    if isinstance(payload, dict):
        for key in ("model_state_dict", "state_dict"):
            if key in payload and isinstance(payload[key], dict):
                return payload[key], payload
    if not isinstance(payload, dict):
        raise TypeError(f"Checkpoint is not a state dictionary: {checkpoint_path}")
    return payload, payload


import hashlib


def compute_sha256(filepath: str) -> str:
    """Calculates SHA-256 hash for strict provenance tracking."""
    sha = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192 * 1024):
            sha.update(chunk)
    return sha.hexdigest()


class BenchmarkModel:
    def __init__(self, name, checkpoint_path, backbone_name, device):
        self.name = name
        self.checkpoint_path = str(checkpoint_path)
        self.backbone_name = backbone_name
        self.device = device
        self.sha256 = compute_sha256(self.checkpoint_path)
        state, self.payload = checkpoint_state(self.checkpoint_path)
        normalized_state = {}
        for key, value in state.items():
            clean_key = key
            for prefix in ("module.", "model."):
                if clean_key.startswith(prefix):
                    clean_key = clean_key[len(prefix):]
            if torch.is_tensor(value):
                normalized_state[clean_key] = value
        head_weight = normalized_state.get("head_c.weight")
        self.num_classes_c = int(head_weight.shape[0]) if head_weight is not None else 1
        if self.num_classes_c not in (1, 3):
            raise ValueError(f"Unsupported Head C width {self.num_classes_c} in {self.checkpoint_path}")
        self.model = MultiTaskChestXrayNet(
            backbone_name=backbone_name,
            pretrained=False,
            num_classes_a=14,
            num_classes_b=1,
            num_classes_c=self.num_classes_c,
        )
        incompatible = self.model.load_state_dict(normalized_state, strict=False)
        loaded_keys = len(self.model.state_dict()) - len(incompatible.missing_keys)
        if loaded_keys < 0.90 * len(self.model.state_dict()):
            raise RuntimeError(
                f"Checkpoint architecture mismatch: loaded {loaded_keys}/{len(self.model.state_dict())} tensors; "
                f"missing={len(incompatible.missing_keys)}, unexpected={len(incompatible.unexpected_keys)}"
            )
        self.model.to(device).eval()
        self.sha256 = compute_sha256(self.checkpoint_path)
        print(f"[+] Loaded {name} from {self.checkpoint_path}")
        print(f"    SHA-256: {self.sha256[:16]}... | Head C width: {self.num_classes_c} | loaded tensors: {loaded_keys}")

    @torch.inference_mode()
    def predict_batch(self, batch_tensors):
        batch_tensors = batch_tensors.to(self.device, non_blocking=True)
        with torch.amp.autocast(device_type=self.device.type, enabled=(self.device.type == "cuda")):
            outputs = self.model(batch_tensors)
        prob_a = torch.sigmoid(outputs["head_a"]).float().cpu().numpy()
        prob_b = torch.sigmoid(outputs["head_b"]).float().cpu().numpy().reshape(-1)
        if self.num_classes_c == 1:
            prob_c = torch.sigmoid(outputs["head_c"]).float().cpu().numpy().reshape(-1)
        else:
            # Legacy 3-logit checkpoints use [Normal, Non-COVID Pneumonia, COVID-19].
            prob_c = torch.softmax(outputs["head_c"], dim=-1)[:, 2].float().cpu().numpy()
        return {"head_a": prob_a, "head_b": prob_b, "head_c": prob_c}


ckpt_definitions = [
    ("DenseNet-121 Baseline", "best_model.pth", "densenet121"),
    ("ResNet-50", "best_resnet50.pth", "resnet50"),
    ("AG-MTCNet Retrained", "ag_mtcnet_best.pth", "densenet121"),
    ("WSOL Fine-tuned", "best_wsol_finetune.pth", "densenet121"),
]


def discover_checkpoint(filename):
    direct = Path(filename)
    if direct.is_file():
        return direct
    exact_matches = glob.glob(f"{k_input}/**/{Path(filename).name}", recursive=True)
    if exact_matches:
        return Path(sorted(exact_matches, key=str.casefold)[0])
    target_stem = Path(filename).stem.casefold()
    stem_matches = []
    for input_root in (Path(k_input), Path("/kaggle/working")):
        if not input_root.is_dir():
            continue
        for root, dirs, files in os.walk(input_root):
            dirs.sort()
            for candidate in files:
                candidate_path = Path(candidate)
                if candidate_path.suffix.casefold() == ".pth" and candidate_path.stem.casefold() == target_stem:
                    stem_matches.append(Path(root) / candidate)
    return sorted(stem_matches, key=str.casefold)[0] if stem_matches else None


loaded_models: List[BenchmarkModel] = []
manifest = {
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "input_root": k_input,
    "device": str(device),
    "datasets": {
        "nih_test_samples": len(nih_samples),
        "tb_test_samples": len(tb_test_samples),
        "covid_test_samples": len(covid_test_samples),
        "covid_manifest": covid_manifest_used,
        "covid_negative_control_source": control_source,
    },
    "models": {},
}
for name, filename, architecture in ckpt_definitions:
    checkpoint_path = discover_checkpoint(filename)
    if checkpoint_path is None:
        print(f"[!] Checkpoint not found: {filename}")
        continue
    try:
        model = BenchmarkModel(name, checkpoint_path, architecture, device)
        loaded_models.append(model)
        manifest["models"][name] = {
            "path": str(checkpoint_path),
            "sha256": model.sha256,
            "bytes": Path(checkpoint_path).stat().st_size,
            "backbone": architecture,
            "head_c_classes": model.num_classes_c,
        }
    except Exception as exc:
        print(f"[!] Could not load {name} from {checkpoint_path}: {exc}")

if not loaded_models:
    raise FileNotFoundError("No compatible .pth checkpoint was found under /kaggle/input or /kaggle/working.")
with open(OUTPUT_DIR / "benchmark_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)
print(f"[+] Saved SHA-256 benchmark provenance to {OUTPUT_DIR / 'benchmark_manifest.json'}")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: Evaluation Execution, Binary Metrics & 95% Bootstrap CI
# ---------------------------------------------------------------------------
def bootstrap_binary_auc_ci(y_true, y_prob, n_bootstraps=500, seed=42):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=np.float64).reshape(-1)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n_bootstraps):
        indices = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue
        scores.append(roc_auc_score(y_true[indices], y_prob[indices]))
    if not scores:
        return float("nan"), float("nan")
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


def bootstrap_macro_auc_ci(y_true, y_prob, n_bootstraps=500, seed=42):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    if y_true.ndim != 2 or y_true.shape != y_prob.shape or len(y_true) == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n_bootstraps):
        indices = rng.integers(0, len(y_true), len(y_true))
        class_aucs = [
            roc_auc_score(y_true[indices, class_idx], y_prob[indices, class_idx])
            for class_idx in range(y_true.shape[1])
            if len(np.unique(y_true[indices, class_idx])) == 2
        ]
        if class_aucs:
            scores.append(float(np.mean(class_aucs)))
    if not scores:
        return float("nan"), float("nan")
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


def binary_operating_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=np.float64).reshape(-1)
    y_pred = (y_prob >= threshold).astype(np.int64)
    _, sensitivity, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    negatives = y_true == 0
    specificity = float(np.mean(y_pred[negatives] == 0)) if np.any(negatives) else float("nan")
    return float(sensitivity), specificity, float(f1)


def test_derived_youden_threshold(y_true, y_prob):
    """Descriptive held-out threshold only; do not use it as a deployment threshold."""
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=np.float64).reshape(-1)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    finite = np.isfinite(thresholds)
    if not np.any(finite):
        return 0.5
    best = np.flatnonzero(finite)[np.argmax((tpr - fpr)[finite])]
    return float(np.clip(thresholds[best], 0.0, 1.0))


# The requested clean ensemble has a fixed membership; never silently change it.
model_by_name = {model.name: model for model in loaded_models}
clean_members = ["ResNet-50", "AG-MTCNet Retrained", "WSOL Fine-tuned"]
all_members = ["DenseNet-121 Baseline", "ResNet-50", "AG-MTCNet Retrained", "WSOL Fine-tuned"]
ensemble_specs = []
if all(name in model_by_name for name in clean_members):
    ensemble_specs.append(("Consensus Ensemble (Clean SOTA)", clean_members))
else:
    print(f"[!] Clean SOTA ensemble omitted; missing: {[name for name in clean_members if name not in model_by_name]}")
if all(name in model_by_name for name in all_members):
    ensemble_specs.append(("Consensus Ensemble (Full 4-Model)", all_members))
else:
    print(f"[!] Full 4-model ensemble omitted; missing: {[name for name in all_members if name not in model_by_name]}")

eval_rosters = [model.name for model in loaded_models] + [name for name, _ in ensemble_specs]
results_summary, per_class_nih, raw_predictions = [], [], []


def collect_task_predictions(loader, task_key, task_name):
    if loader is None or len(loader) == 0:
        return None, None, None
    true_batches, filename_batches = [], []
    pred_batches = {name: [] for name in eval_rosters}
    for tensors, labels, filenames in tqdm(loader, desc=f"{task_name} inference"):
        true_batches.append(labels.numpy())
        filename_batches.extend(list(filenames))
        batch_probs = {}
        for model in loaded_models:
            probability = model.predict_batch(tensors)[task_key]
            pred_batches[model.name].append(probability)
            batch_probs[model.name] = probability
        for ensemble_name, members in ensemble_specs:
            pred_batches[ensemble_name].append(np.mean([batch_probs[name] for name in members], axis=0))
    y_true = np.concatenate(true_batches, axis=0)
    predictions = {name: np.concatenate(parts, axis=0) for name, parts in pred_batches.items()}
    return y_true, predictions, filename_batches


# Head A: per-pathology AUROC and patient-resampled macro-AUROC confidence intervals.
y_true_a, predictions_a, filenames_a = collect_task_predictions(nih_loader, "head_a", "Head A / NIH")
if y_true_a is not None:
    y_true_a = np.asarray(y_true_a, dtype=np.float32).reshape(-1, len(NIH_14_DISEASES))
    for model_name, y_pred_a in predictions_a.items():
        class_aucs = []
        for class_idx, disease in enumerate(NIH_14_DISEASES):
            labels, probabilities = y_true_a[:, class_idx], y_pred_a[:, class_idx]
            auc_value = float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else float("nan")
            class_aucs.append(auc_value)
            per_class_nih.append({
                "Model": model_name,
                "Pathology": disease,
                "AUROC": auc_value,
                "Positive_Support": int(labels.sum()),
                "Total_Evaluated": len(labels),
            })
        valid_aucs = [value for value in class_aucs if np.isfinite(value)]
        macro_auc = float(np.mean(valid_aucs)) if valid_aucs else float("nan")
        ci_low, ci_high = bootstrap_macro_auc_ci(y_true_a, y_pred_a, n_bootstraps=500)
        predicted = (y_pred_a >= 0.5).astype(np.float32)
        f1_macro = float(f1_score(y_true_a, predicted, average="macro", zero_division=0))
        sensitivities, specificities = [], []
        for class_idx in range(len(NIH_14_DISEASES)):
            labels, decisions = y_true_a[:, class_idx], predicted[:, class_idx]
            if len(np.unique(labels)) < 2:
                continue
            _, sensitivity, _, _ = precision_recall_fscore_support(labels, decisions, average="binary", zero_division=0)
            negatives = labels == 0
            sensitivities.append(float(sensitivity))
            specificities.append(float(np.mean(decisions[negatives] == 0)) if np.any(negatives) else float("nan"))
        results_summary.append({
            "Model": model_name, "Task": "Head A (NIH-14)",
            "Macro_AUROC": macro_auc, "95%_CI": f"[{ci_low:.4f}, {ci_high:.4f}]",
            "Macro_F1": f1_macro,
            "Sensitivity": float(np.nanmean(sensitivities)) if sensitivities else float("nan"),
            "Specificity": float(np.nanmean(specificities)) if specificities else float("nan"),
            "Total_Samples": len(y_true_a),
        })
        for filename, truth, probability in zip(filenames_a, y_true_a, y_pred_a):
            raw_predictions.append({
                "Model": model_name, "Task": "Head A (NIH-14)", "Filename": filename,
                "Ground_Truth": json.dumps(truth.astype(int).tolist()),
                "Probabilities": json.dumps(np.round(probability, 7).tolist()),
            })

# Head B and Head C: binary AUROC, fixed 0.5 operating point, and descriptive test-set Youden point.
for task_key, task_name, loader in (
    ("head_b", "Head B (TB Triage)", tb_loader),
    ("head_c", "Head C (COVID Screening)", covid_loader),
):
    y_true, predictions, filenames = collect_task_predictions(loader, task_key, task_name)
    if y_true is None:
        continue
    y_true = np.asarray(y_true, dtype=np.int64).reshape(-1)
    for model_name, probabilities in predictions.items():
        probabilities = np.asarray(probabilities, dtype=np.float64).reshape(-1)
        auc_value = float(roc_auc_score(y_true, probabilities)) if len(np.unique(y_true)) == 2 else float("nan")
        ci_low, ci_high = bootstrap_binary_auc_ci(y_true, probabilities, n_bootstraps=500)
        sens_05, spec_05, f1_05 = binary_operating_metrics(y_true, probabilities, 0.5)
        youden_threshold = test_derived_youden_threshold(y_true, probabilities)
        if np.isfinite(youden_threshold):
            sens_youden, spec_youden, f1_youden = binary_operating_metrics(y_true, probabilities, youden_threshold)
        else:
            sens_youden = spec_youden = f1_youden = float("nan")
        results_summary.append({
            "Model": model_name, "Task": task_name,
            "Macro_AUROC": auc_value, "95%_CI": f"[{ci_low:.4f}, {ci_high:.4f}]",
            "Sensitivity": sens_05, "Specificity": spec_05, "F1": f1_05,
            "Threshold_0.5": 0.5, "Youden_Threshold_Test_Descriptive": youden_threshold,
            "Sensitivity_Youden": sens_youden, "Specificity_Youden": spec_youden,
            "F1_Youden": f1_youden, "Total_Samples": len(y_true),
        })
        for filename, truth, probability in zip(filenames, y_true, probabilities):
            raw_predictions.append({
                "Model": model_name, "Task": task_name, "Filename": filename,
                "Ground_Truth": int(truth), "Probability_COVID_or_TB": float(probability),
            })

df_summary = pd.DataFrame(results_summary)
df_per_class = pd.DataFrame(per_class_nih)
df_summary.to_csv(OUTPUT_DIR / "classification_metrics.csv", index=False)
df_per_class.to_csv(OUTPUT_DIR / "nih_per_class_auc.csv", index=False)
raw_predictions_path = OUTPUT_DIR / "test_predictions_raw.csv.gz"
pd.DataFrame(raw_predictions).to_csv(raw_predictions_path, index=False, compression="gzip")
print(f"[+] Metrics exported under {OUTPUT_DIR}")
print(f"[+] Raw per-sample predictions exported to {raw_predictions_path}")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: Weakly-Supervised Object Localization (RSNA DICOM Box-IoU)
# ---------------------------------------------------------------------------
def compute_box_iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = min(box_a[0], box_a[2]), min(box_a[1], box_a[3]), max(box_a[0], box_a[2]), max(box_a[1], box_a[3])
    xb1, yb1, xb2, yb2 = min(box_b[0], box_b[2]), min(box_b[1], box_b[3]), max(box_b[0], box_b[2]), max(box_b[1], box_b[3])
    left, top, right, bottom = max(xa1, xb1), max(ya1, yb1), min(xa2, xb2), min(ya2, yb2)
    if right <= left or bottom <= top:
        return 0.0
    intersection = (right - left) * (bottom - top)
    union = (xa2 - xa1) * (ya2 - ya1) + (xb2 - xb1) * (yb2 - yb1) - intersection
    return float(np.clip(intersection / max(union, 1e-8), 0.0, 1.0))


def extract_bboxes_from_mask(mask, threshold=0.35, min_area=100):
    binary = (np.asarray(mask) >= threshold).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for contour in contours:
        if cv2.contourArea(contour) >= min_area:
            x, y, width, height = cv2.boundingRect(contour)
            boxes.append([float(x), float(y), float(x + width), float(y + height)])
    return boxes


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.hook = target_layer.register_forward_hook(self._capture_activation)

    def _capture_activation(self, module, inputs, output):
        self.activations = output
        output.register_hook(self._capture_gradient)

    def _capture_gradient(self, gradient):
        self.gradients = gradient

    def generate(self, tensor, class_idx=6):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        self.activations = self.gradients = None
        logits = self.model(tensor)["head_a"]
        logits[:, class_idx].sum().backward()
        if self.activations is None or self.gradients is None:
            return np.zeros(tensor.shape[-2:], dtype=np.float32)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0].detach().float().cpu().numpy()
        maximum = float(cam.max()) if cam.size else 0.0
        return (cam / maximum).astype(np.float32) if maximum > 1e-8 else np.zeros_like(cam, dtype=np.float32)

    def remove_hook(self):
        self.hook.remove()


def read_rsna_dicom(path):
    import pydicom
    ds = pydicom.dcmread(path)
    image = ds.pixel_array.astype(np.float32)
    image = image * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        raise ValueError(f"No finite pixels in {path}")
    low, high = np.percentile(finite, [0.5, 99.5])
    if high <= low:
        high = low + 1.0
    image = np.clip((image - low) / (high - low), 0.0, 1.0)
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
        image = 1.0 - image
    return np.round(image * 255.0).astype(np.uint8)


rsna_label_candidates = sorted(set(
    glob.glob(f"{k_input}/**/stage_2_train_labels.csv", recursive=True)
    + glob.glob(f"{k_input}/**/stage_1_train_labels.csv", recursive=True)
))
rsna_label_path = next((path for path in rsna_label_candidates if os.path.isfile(path)), None)
rsna_file_index = build_filename_index(rsna_candidates or [k_input], extensions=(".dcm",))
wsol_metrics = []

if rsna_label_path and rsna_file_index:
    rsna_frame = pd.read_csv(rsna_label_path)
    required_columns = {"patientId", "Target", "x", "y", "width", "height"}
    if not required_columns.issubset(rsna_frame.columns):
        print(f"[!] RSNA labels do not contain required columns: {sorted(required_columns)}; WSOL omitted.")
    else:
        positive_rows = rsna_frame[
            (pd.to_numeric(rsna_frame["Target"], errors="coerce") == 1)
            & rsna_frame[["x", "y", "width", "height"]].notna().all(axis=1)
        ].copy()
        positive_patient_ids = positive_rows["patientId"].astype(str).drop_duplicates().tolist()[:100]
        print(f"[*] RSNA annotated positive studies available: {len(positive_patient_ids)} (maximum 100).")
        for benchmark_model in loaded_models:
            backbone = benchmark_model.model.backbone
            if benchmark_model.backbone_name == "densenet121":
                target_layer = backbone.features.denseblock4
            else:
                target_layer = backbone.layer4
            explainer = GradCAM(benchmark_model.model, target_layer)
            study_ious = []
            try:
                for patient_id in tqdm(positive_patient_ids, desc=f"RSNA WSOL / {benchmark_model.name}"):
                    dicom_path = rsna_file_index.get(f"{patient_id}.dcm".casefold())
                    if not dicom_path:
                        continue
                    try:
                        gray = read_rsna_dicom(dicom_path)
                        height, width = gray.shape[:2]
                        rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
                        resized = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                        normalized = (resized.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
                        tensor = torch.from_numpy(normalized.transpose(2, 0, 1).copy()).unsqueeze(0).to(device)
                    except Exception as exc:
                        print(f"[!] Skipping unreadable DICOM {dicom_path}: {exc}")
                        continue
                    cam = explainer.generate(tensor, class_idx=6)
                    predicted_boxes = extract_bboxes_from_mask(cam, threshold=0.35, min_area=100)
                    rows = positive_rows[positive_rows["patientId"].astype(str) == patient_id]
                    sx, sy = IMG_SIZE / float(width), IMG_SIZE / float(height)
                    true_boxes = [
                        [float(row.x) * sx, float(row.y) * sy,
                         (float(row.x) + float(row.width)) * sx,
                         (float(row.y) + float(row.height)) * sy]
                        for row in rows.itertuples(index=False)
                    ]
                    if not true_boxes:
                        continue
                    study_ious.append(max(
                        (compute_box_iou(pred_box, true_box) for pred_box in predicted_boxes for true_box in true_boxes),
                        default=0.0,
                    ))
            finally:
                explainer.remove_hook()
            if study_ious:
                mean_iou = float(np.mean(study_ious))
                success_30 = 100.0 * float(np.mean(np.asarray(study_ious) >= 0.30))
                success_50 = 100.0 * float(np.mean(np.asarray(study_ious) >= 0.50))
                wsol_metrics.append({
                    "Model": benchmark_model.name, "Dataset": "RSNA Pneumonia",
                    "Mean_Box_IoU": mean_iou,
                    "Success_Rate_IoU_30": success_30,
                    "Success_Rate_IoU_50": success_50,
                    "Evaluated_Cases": len(study_ious),
                })
                print(f" -> {benchmark_model.name}: Mean Box-IoU={mean_iou:.4f}; @0.30={success_30:.1f}%; @0.50={success_50:.1f}%")
else:
    print("[!] RSNA labels or DICOM image index not found; WSOL evaluation omitted.")

df_wsol = pd.DataFrame(wsol_metrics)
df_wsol.to_csv(OUTPUT_DIR / "wsol_iou_metrics.csv", index=False)
print(f"[+] WSOL metrics exported to {OUTPUT_DIR / 'wsol_iou_metrics.csv'}")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: Publication-Ready Monochrome Visualizations (300 DPI)
# ---------------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "font.size": 10, "figure.autolayout": True})
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Figure 1: AUROC for each model and task.
if not df_summary.empty:
    fig, ax = plt.subplots(figsize=(11, 6), dpi=300)
    pivot = df_summary.pivot(index="Task", columns="Model", values="Macro_AUROC")
    pivot.plot(kind="bar", ax=ax, width=0.82, colormap="Greys", edgecolor="black", linewidth=0.6)
    ax.set_title("Multi-Task AUROC Comparative Benchmark", fontsize=12, fontweight="bold")
    ax.set_ylabel("AUROC")
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel("")
    ax.grid(axis="y", linestyle="--", color="0.75", alpha=0.7)
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    plt.xticks(rotation=0)
    fig.savefig(FIGURES_DIR / "multi_model_auroc_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Figure 2: NIH per-pathology AUROC.
if not df_per_class.empty:
    fig, ax = plt.subplots(figsize=(13, 7), dpi=300)
    pivot_nih = df_per_class.pivot(index="Pathology", columns="Model", values="AUROC")
    sns.heatmap(pivot_nih, annot=True, fmt=".3f", cmap="Greys", vmin=0.0, vmax=1.0,
                linewidths=0.35, linecolor="0.65", cbar_kws={"label": "AUROC"}, ax=ax)
    ax.set_title("Head A: Per-Pathology AUROC on NIH ChestX-ray14", fontsize=12, fontweight="bold")
    ax.set_xlabel("Model")
    ax.set_ylabel("Pathology")
    fig.savefig(FIGURES_DIR / "nih_per_pathology_auroc.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Figure 3: IoU and success rates share a 0-1 scale.
if not df_wsol.empty:
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)
    plot_data = df_wsol.set_index("Model")[["Mean_Box_IoU", "Success_Rate_IoU_30", "Success_Rate_IoU_50"]].copy()
    plot_data["Success_Rate_IoU_30"] = plot_data["Success_Rate_IoU_30"] / 100.0
    plot_data["Success_Rate_IoU_50"] = plot_data["Success_Rate_IoU_50"] / 100.0
    plot_data.columns = ["Mean Box-IoU", "Success rate at IoU 0.30", "Success rate at IoU 0.50"]
    plot_data.plot(kind="bar", ax=ax, width=0.82, color=["0.15", "0.48", "0.78"], edgecolor="black", linewidth=0.6)
    # Grayscale-only rendering for the report standard.
    for patch, shade in zip(ax.patches, ["0.15", "0.48", "0.78"] * len(plot_data)):
        patch.set_facecolor(shade)
    ax.set_title("RSNA Pneumonia Localization Benchmark", fontsize=12, fontweight="bold")
    ax.set_ylabel("Score / Proportion")
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("")
    ax.grid(axis="y", linestyle="--", color="0.75", alpha=0.7)
    ax.legend(frameon=True)
    plt.xticks(rotation=0)
    fig.savefig(FIGURES_DIR / "wsol_box_iou_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
print(f"[+] Publication figures saved to {FIGURES_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 7: Evidence-Based Decision Matrix (CV Evidence vs Further Fine-Tuning)
# ---------------------------------------------------------------------------
def print_booktabs(frame):
    if frame.empty:
        print("No metric rows are available.")
        return
    headers = list(frame.columns)
    print("\\toprule")
    print(" & ".join(str(value).replace("_", r"\_") for value in headers) + r" \\")
    print("\\midrule")
    for row in frame.itertuples(index=False, name=None):
        cells = []
        for value in row:
            if isinstance(value, (float, np.floating)):
                cells.append("--" if not np.isfinite(value) else f"{value:.4f}")
            else:
                cells.append(str(value).replace("_", r"\_"))
        print(" & ".join(cells) + r" \\")
    print("\\bottomrule")


print("=== MASTER BENCHMARK ENGINE V2 SUMMARY ===")
print_booktabs(df_summary[[col for col in ("Model", "Task", "Macro_AUROC", "95%_CI", "Sensitivity", "Specificity", "F1", "Samples") if col in df_summary.columns]])
if not df_wsol.empty:
    print("\n=== WSOL LOCALIZATION (RSNA) ===")
    print_booktabs(df_wsol)


def best_metric(task_name, column="Macro_AUROC"):
    rows = df_summary[df_summary["Task"] == task_name] if not df_summary.empty else pd.DataFrame()
    values = pd.to_numeric(rows[column], errors="coerce") if column in rows.columns else pd.Series(dtype=float)
    values = values[np.isfinite(values)]
    return float(values.max()) if not values.empty else float("nan")


best_nih = best_metric("Head A (NIH-14)")
best_tb = best_metric("Head B (TB Triage)")
best_covid = best_metric("Head C (COVID Screening)")
best_iou = float(df_wsol["Mean_Box_IoU"].max()) if not df_wsol.empty else float("nan")
criteria = {
    "Head A NIH Macro AUROC >= 0.82": (best_nih, 0.82),
    "Head B TB AUROC >= 0.90": (best_tb, 0.90),
    "Head C COVID AUROC >= 0.90": (best_covid, 0.90),
    "RSNA Mean Box-IoU >= 0.25": (best_iou, 0.25),
}
print("\n=== ACCEPTANCE CRITERIA ===")
for criterion, (value, target) in criteria.items():
    state = "MEETS" if np.isfinite(value) and value >= target else "NEEDS FOLLOW-UP"
    rendered = f"{value:.4f}" if np.isfinite(value) else "not measured"
    print(f"{criterion}: {rendered} (target {target:.2f}) — {state}")

cv_ready = all(np.isfinite(value) and value >= target for value, target in criteria.values())
if cv_ready:
    print("\nRecommendation: the requested benchmark thresholds are met; summarize the measured work in the CV with its dataset and evaluation limits.")
else:
    unmet = [name for name, (value, target) in criteria.items() if not np.isfinite(value) or value < target]
    print("\nRecommendation: continue targeted validation or fine-tuning for the unmet criteria:")
    for name in unmet:
        print(f" - {name}")
print("A missing metric is treated as unmet; no unavailable result is inferred as a success.")
